# M2 Lab — RAG Fundamentals & Retrieval Patterns

**Track:** Applied Agent Engineering Foundations  
**Module:** M2 — RAG Fundamentals & Retrieval Patterns  
**Environment:** SageMaker Jupyter Notebook  
**Assumption:** Relevant enterprise documents are already available in S3

## What learners will learn
1. Read enterprise documents from S3.
2. Compare chunking strategies.
3. Generate embeddings and retrieve top matches.
4. Use metadata filters.
5. Build a grounded RAG response.
6. Add reranking.
7. Complete a mini-hack on retrieval quality.

> **Explain to learners:** RAG quality is mostly a pipeline design problem. Better chunking, metadata, filtering, and reranking usually matter more than just swapping the LLM.


## Instructor note on the original notebook

The original M2 notebook mixed together:
- S3 security checks
- RAG construction
- agent tool wrapping
- MCP / orchestration
- governance / observability

This classroom-ready version keeps **M2 focused on retrieval and grounded answer generation**. Agentic orchestration and governance are moved into later modules.


In [ ]:

# Uncomment only if your environment needs these libraries
# %pip install -q boto3 pandas numpy python-docx PyPDF2

import json
from io import BytesIO
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
import PyPDF2
from docx import Document

AWS_REGION = boto3.Session().region_name or "us-east-1"
S3_BUCKET = "YOUR_S3_BUCKET"
S3_PREFIX = "YOUR_DOCUMENT_PREFIX/"   # Example: hr_docs/
EMBED_MODEL_ID = "amazon.titan-embed-text-v1"
TEXT_MODEL_ID = "amazon.nova-lite-v1:0"

s3 = boto3.client("s3", region_name=AWS_REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)

print("Region:", AWS_REGION)
print("Bucket:", S3_BUCKET)
print("Prefix:", S3_PREFIX)


## Step 1 — Optional S3 access-control demo

Use this section only if you want to demonstrate enterprise access controls.
The aim is to show learners that RAG systems must respect data boundaries.

**Expected teaching point:** Read access may be allowed, while write, delete, or copy actions may be blocked by policy.


In [ ]:

# Replace with an actual object key that exists in your S3 prefix if you want to demo this.
SAMPLE_OBJECT_KEY = f"{S3_PREFIX}sample.csv"

# Example read check
# obj = s3.get_object(Bucket=S3_BUCKET, Key=SAMPLE_OBJECT_KEY)
# print("Read check passed. Bytes:", len(obj["Body"].read()))

# Example write check (commented by default)
# import io
# df = pd.DataFrame({"demo":[1,2,3]})
# csv_buffer = io.StringIO()
# df.to_csv(csv_buffer, index=False)
# s3.put_object(Bucket=S3_BUCKET, Key=f"{S3_PREFIX}write_test.csv", Body=csv_buffer.getvalue())


## Step 2 — Load documents from S3

This loader handles:
- `.txt`
- `.docx`
- `.pdf`

**Discuss with learners:**
- Why extraction quality matters
- Why PDF extraction may be messy
- Why enterprise ingestion pipelines often need preprocessing


In [ ]:

def read_txt_from_s3(bucket: str, key: str) -> str:
    response = s3.get_object(Bucket=bucket, Key=key)
    return response["Body"].read().decode("utf-8", errors="ignore")

def read_docx_from_s3(bucket: str, key: str) -> str:
    response = s3.get_object(Bucket=bucket, Key=key)
    stream = BytesIO(response["Body"].read())
    doc = Document(stream)
    return "\n".join([p.text for p in doc.paragraphs])

def read_pdf_from_s3(bucket: str, key: str) -> str:
    response = s3.get_object(Bucket=bucket, Key=key)
    stream = BytesIO(response["Body"].read())
    reader = PyPDF2.PdfReader(stream)
    pages = [page.extract_text() or "" for page in reader.pages]
    return "\n".join(pages)

def extract_text_from_s3(bucket: str, key: str) -> str:
    key_lower = key.lower()
    if key_lower.endswith(".txt"):
        return read_txt_from_s3(bucket, key)
    if key_lower.endswith(".docx"):
        return read_docx_from_s3(bucket, key)
    if key_lower.endswith(".pdf"):
        return read_pdf_from_s3(bucket, key)
    return ""

def list_document_keys(bucket: str, prefix: str) -> list[str]:
    paginator = s3.get_paginator("list_objects_v2")
    keys = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.lower().endswith((".txt", ".docx", ".pdf")):
                keys.append(key)
    return keys

document_keys = list_document_keys(S3_BUCKET, S3_PREFIX)
print("Document count:", len(document_keys))
print(document_keys[:10])


In [ ]:

documents = {}

for key in document_keys:
    text = extract_text_from_s3(S3_BUCKET, key)
    if text.strip():
        documents[key] = text

print("Loaded documents:", len(documents))
for key in list(documents.keys())[:5]:
    print("-", key, "| chars:", len(documents[key]))


## Step 3 — Compare chunking strategies

We will compare:
1. Basic fixed chunks
2. Fixed chunks with overlap
3. Section-based chunks

**Explain to learners:** Chunking is a design choice. There is no universal best chunking strategy.


In [ ]:

def basic_chunking(text: str, chunk_size: int = 500) -> list[str]:
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

def chunk_with_overlap(text: str, chunk_size: int = 500, overlap: int = 100) -> list[str]:
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(text), step):
        chunks.append(text[i:i+chunk_size])
    return chunks

def section_based_chunking(text: str, min_section_chars: int = 120) -> list[str]:
    sections = [sec.strip() for sec in text.split("\n\n")]
    return [sec for sec in sections if len(sec) >= min_section_chars]


In [ ]:

sample_key = next(iter(documents))
sample_text = documents[sample_key]

comparison = {
    "basic": basic_chunking(sample_text),
    "overlap": chunk_with_overlap(sample_text),
    "section": section_based_chunking(sample_text),
}

for method, chunks in comparison.items():
    print("=" * 80)
    print("METHOD:", method)
    print("NUMBER OF CHUNKS:", len(chunks))
    print("FIRST CHUNK PREVIEW:")
    print(chunks[0][:500] if chunks else "No chunks")
    print()


## Checkpoint discussion

Ask learners:
- Which method is easiest to implement?
- Which method is likely to preserve meaning better?
- Where might overlap help?
- Why might sections outperform fixed windows in policy documents?


## Step 4 — Build chunk records with metadata

We will keep:
- chunk text
- source file
- chunk id
- department / topic metadata inferred from file name

**Explain to learners:** Metadata filtering is one of the easiest ways to improve retrieval quality in enterprise systems.


In [ ]:

def infer_department_from_key(key: str) -> str:
    key_lower = key.lower()
    if "hr" in key_lower:
        return "HR"
    if "finance" in key_lower:
        return "Finance"
    if "it" in key_lower or "tech" in key_lower:
        return "IT"
    return "General"

chunked_data = []

for key, text in documents.items():
    chunks = chunk_with_overlap(text, chunk_size=600, overlap=120)
    for i, chunk in enumerate(chunks):
        chunked_data.append({
            "text": chunk,
            "source": key,
            "chunk_id": i,
            "department": infer_department_from_key(key)
        })

print("Total chunk records:", len(chunked_data))
pd.DataFrame(chunked_data).head()


## Step 5 — Generate embeddings

This cell uses Titan text embeddings.
Each chunk gets a vector so that we can compare the query and the chunk in vector space.


In [ ]:

def get_embedding(text: str) -> list[float]:
    response = bedrock_runtime.invoke_model(
        modelId=EMBED_MODEL_ID,
        body=json.dumps({"inputText": text})
    )
    payload = json.loads(response["body"].read())
    return payload["embedding"]

# Warning:
# On a large corpus, embed in batches or cache results.
# For classroom use, run this on a smaller subset first if needed.

max_chunks_for_demo = min(len(chunked_data), 40)

for item in chunked_data[:max_chunks_for_demo]:
    item["embedding"] = get_embedding(item["text"])

print("Embedded chunk count for demo:", len([x for x in chunked_data if "embedding" in x]))


## Step 6 — Retrieval by similarity

We use cosine similarity for the classroom demo.


In [ ]:

def cosine_similarity(a, b) -> float:
    a = np.array(a)
    b = np.array(b)
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

def retrieve_top_k(query: str, k: int = 3) -> list[tuple]:
    query_embedding = get_embedding(query)
    scores = []

    for item in chunked_data:
        if "embedding" not in item:
            continue
        score = cosine_similarity(query_embedding, item["embedding"])
        scores.append((score, item))

    scores.sort(key=lambda x: x[0], reverse=True)
    return scores[:k]

def retrieve_with_filter(query: str, department: str | None = None, k: int = 3) -> list[tuple]:
    query_embedding = get_embedding(query)
    scores = []

    for item in chunked_data:
        if "embedding" not in item:
            continue
        if department and item["department"] != department:
            continue
        score = cosine_similarity(query_embedding, item["embedding"])
        scores.append((score, item))

    scores.sort(key=lambda x: x[0], reverse=True)
    return scores[:k]


In [ ]:

query = "What is the leave policy?"
results = retrieve_top_k(query, k=3)

for score, item in results:
    print("=" * 80)
    print("Score:", round(score, 4))
    print("Source:", item["source"])
    print("Department:", item["department"])
    print(item["text"][:700])
    print()


## Step 7 — Build grounded answers

Now retrieval results become context for answer generation.

**Prompt rule:** If the answer is not in the retrieved context, say `"I don't know."`


In [ ]:

def build_context(results: list[dict]) -> str:
    context = []
    for item in results:
        context.append(f"[Source: {item['source']}]\n{item['text']}")
    return "\n\n".join(context)

def generate_response(query: str, context: str) -> str:
    prompt = f'''
You are an enterprise AI assistant.

Answer ONLY from the provided context.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{query}
'''
    response = bedrock_runtime.converse(
        modelId=TEXT_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 300, "temperature": 0.2}
    )
    return response["output"]["message"]["content"][0]["text"]


In [ ]:

retrieved_items = [item for _, item in results]
context = build_context(retrieved_items)
answer = generate_response(query, context)

print("QUESTION:", query)
print("\nANSWER:")
print(answer)


## Step 8 — Add reranking

Vector retrieval gets candidates quickly. Reranking is a second pass to reorder them more intelligently.

**Explain to learners:** Retrieval finds possible matches. Reranking tries to sort the best ones to the top.


In [ ]:

import re

def rerank_results_llm(query: str, results: list[tuple], top_k: int = 3) -> list[dict]:
    docs = ""
    for i, (score, item) in enumerate(results):
        docs += f"\nDocument {i}:\n{item['text']}\n"

    prompt = f'''
You are a reranking assistant.

Given a query and multiple documents, return ONLY a Python-style list
of the most relevant document numbers in order of relevance.

Query:
{query}

Documents:
{docs}

Return only something like:
[2, 0, 1]
'''

    response = bedrock_runtime.converse(
        modelId=TEXT_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 100, "temperature": 0}
    )

    output = response["output"]["message"]["content"][0]["text"]
    indices = list(map(int, re.findall(r"\d+", output)))

    reranked = []
    for idx in indices[:top_k]:
        if idx < len(results):
            score, item = results[idx]
            reranked.append({
                "text": item["text"],
                "source": item["source"],
                "score": score,
                "department": item["department"]
            })
    return reranked


In [ ]:

candidate_results = retrieve_top_k("What is eligibility for remote work?", k=5)
reranked = rerank_results_llm("What is eligibility for remote work?", candidate_results, top_k=3)

for item in reranked:
    print("=" * 80)
    print("Source:", item["source"])
    print("Score:", round(item["score"], 4))
    print(item["text"][:700])


## Step 9 — Final RAG pipeline


In [ ]:

def rag_pipeline(query: str, department: str | None = None, retrieve_k: int = 5, final_k: int = 3) -> dict:
    if department:
        retrieved = retrieve_with_filter(query, department=department, k=retrieve_k)
    else:
        retrieved = retrieve_top_k(query, k=retrieve_k)

    reranked = rerank_results_llm(query, retrieved, top_k=final_k)
    context = build_context(reranked)
    answer = generate_response(query, context)

    return {
        "query": query,
        "answer": answer,
        "retrieved": retrieved,
        "reranked": reranked
    }

demo = rag_pipeline("What is the leave policy?", department="HR")
print(demo["answer"])


## Mini-hack — Improve retrieval quality

### Task
Build a stronger enterprise policy assistant.

### Minimum requirements
1. Add one more metadata field.
2. Support one filtered query path.
3. Show one query where filtering helps.
4. Show one query where overlap chunking helps.

### Good mini-hack examples
- HR policy assistant
- IT access policy assistant
- Finance compliance policy assistant

### Stretch goals
- add simple citations in final answer
- compare chunking strategies quantitatively
- cache embeddings to a file for faster reruns


In [ ]:

# --- STUDENT WORK AREA ---
# Suggested:
# 1. Add a new metadata field such as document_type or region.
# 2. Update your retrieval function to use that metadata.
# 3. Compare the answer quality before and after the change.


## Wrap-up

Learners should now understand:
- why ingestion and chunking matter
- how embeddings power semantic retrieval
- how metadata improves precision
- how reranking strengthens final answer quality
- why grounded prompting reduces hallucination risk
